In [1]:
# ================================================
# 06_random_forest_final.ipynb - COLDTRACK
# Modelo Final de Mantenimiento Predictivo
# ================================================

import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score
import warnings

warnings.filterwarnings('ignore')

print("=== RANDOM FOREST FINAL - COLDTRACK ===\n")

# ====================== CONFIGURACIÓN DE RUTA ======================
BASE_PATH = "/Users/josea/Documents/ia_data/ColdTrack"
os.chdir(BASE_PATH)
print(f"Directorio de trabajo: {os.getcwd()}\n")

# ====================== 1. CARGAR DATOS ======================
print("Cargando datos procesados...")

X = pd.read_csv("X_features.csv")
y = pd.read_csv("y_target.csv").squeeze()

print(f"Dataset cargado: {X.shape[0]:,} registros × {X.shape[1]} features\n")

# ====================== 2. DIVISIÓN TEMPORAL ======================
tscv = TimeSeriesSplit(n_splits=5)
train_idx, test_idx = list(tscv.split(X))[-1]

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train: {len(X_train):,} registros | Test: {len(X_test):,} registros\n")

# ====================== 3. ENTRENAR MODELO ======================
print("Entrenando modelo Random Forest...")

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
print("Modelo entrenado correctamente.\n")

# ====================== 4. EVALUACIÓN ======================
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("="*70)
print("RESULTADOS DEL MODELO FINAL")
print("="*70)
print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_test, y_proba):.4f}\n")

print("Reporte de Clasificación:")
print(classification_report(y_test, y_pred, target_names=['Normal (0)', 'Mantenimiento (1)']))

print("\nMatriz de Confusión:")
print(confusion_matrix(y_test, y_pred))

# ====================== 5. IMPORTANCIA DE FEATURES ======================
importancia = pd.DataFrame({
    'Feature': X.columns,
    'Importancia': model.feature_importances_
}).sort_values('Importancia', ascending=False)

print("\nTop 10 Features más importantes:")
print(importancia.head(10))

# ====================== 6. GUARDAR MODELO FINAL ======================
joblib.dump(model, "models/coldtrack_rf_model_final.pkl")
joblib.dump(X.columns.tolist(), "models/feature_columns_final.pkl")

print("\n✅ Modelo guardado correctamente en 'models/coldtrack_rf_model_final.pkl'")
print("✅ Columnas guardadas en 'models/feature_columns_final.pkl'")
print("\n¡Modelo Final listo para producción!")

=== RANDOM FOREST FINAL - COLDTRACK ===

Directorio de trabajo: /Users/josea/Documents/ia_data/ColdTrack

Cargando datos procesados...
Dataset cargado: 311,040 registros × 25 features

Train: 259,200 registros | Test: 51,840 registros

Entrenando modelo Random Forest...
Modelo entrenado correctamente.

RESULTADOS DEL MODELO FINAL
Accuracy : 0.9999
ROC AUC  : 1.0000

Reporte de Clasificación:
                   precision    recall  f1-score   support

       Normal (0)       1.00      1.00      1.00     47722
Mantenimiento (1)       1.00      1.00      1.00      4118

         accuracy                           1.00     51840
        macro avg       1.00      1.00      1.00     51840
     weighted avg       1.00      1.00      1.00     51840


Matriz de Confusión:
[[47719     3]
 [    0  4118]]

Top 10 Features más importantes:
                Feature  Importancia
4      ConsumoElectrico     0.229428
5             Vibration     0.200195
21  compressor_overwork     0.158993
19   critical